# Environment setup
We begin by mounting Google Drive and authenticating the environment to securely access proprietary datasets and the World Bank Logistics Performance Index (LPI) stored in Google Sheets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import auth
import gspread
from google.auth import compute_engine
import pandas as pd

auth.authenticate_user()
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

spreadsheet = gc.open('LPI_worldbank')

worksheet_2023 = spreadsheet.worksheet('2023')
df_2023 = pd.DataFrame(worksheet_2023.get_all_records())
df_2023 = df_2023[['Economy', 'LPI Score']].copy()
df_2023.columns = ['country', 'lpi_score']
df_2023['year'] = 2023

old_years = [2018, 2016, 2014, 2012, 2010, 2007]
old_dfs = []

for yr in old_years:
    ws = spreadsheet.worksheet(str(yr))
    data = ws.get_all_values()
    df = pd.DataFrame(data[3:], columns=data[2])

    df = df.iloc[:, [0, 2]].copy()
    df.columns = ['country', 'lpi_score']
    df['year'] = yr
    old_dfs.append(df)

df_lpi = pd.concat([df_2023] + old_dfs, ignore_index=True)
df_lpi['lpi_score'] = pd.to_numeric(df_lpi['lpi_score'], errors='coerce')
df_lpi = df_lpi.dropna(subset=['country', 'lpi_score'])

print(df_lpi.head(10))

                country  lpi_score  year
0             Singapore        4.3  2023
1               Finland        4.2  2023
2               Denmark        4.1  2023
3               Germany        4.1  2023
4           Netherlands        4.1  2023
5           Switzerland        4.1  2023
6               Austria        4.0  2023
7               Belgium        4.0  2023
8                Canada        4.0  2023
9  Hong Kong SAR, China        4.0  2023


#Folder containing files
four files:
WORLD WIDE GOVERNANCE INDICATORS:
1. wb_pv_est- Political Violence Estimate
2. ge_est- Government Effectiveness Estimate
WORLD DEVELOPMENT INDICATORS:
1. inflation_gpd- inflation consumer prices and gdp growth annual both combined
2. LPI_worldbank


In [ ]:
import os

folder = '/content/drive/MyDrive/4files_world_bank'
for f in os.listdir(folder):
    print(f)

inflation_gdp - 2636c3fe-fada-4e46-abb0-14bbbe9dc60f_Data.csv.csv
LPI_worldbank - 2018.csv
wb_pv_est - 9f6967d8-0bd5-4748-8a6e-399a111edaba_Data.csv.csv
ge_est - 0a14da3c-c25f-42f2-b620-dea75446bad0_Data.csv.csv


In [ ]:
import pandas as pd
import os

folder = '/content/drive/MyDrive/4files_world_bank'

df_pv = pd.read_csv(os.path.join(folder, 'wb_pv_est - 9f6967d8-0bd5-4748-8a6e-399a111edaba_Data.csv.csv'), skiprows=4)
df_ge = pd.read_csv(os.path.join(folder, 'ge_est - 0a14da3c-c25f-42f2-b620-dea75446bad0_Data.csv.csv'), skiprows=4)
df_inf_gdp = pd.read_csv(os.path.join(folder, 'inflation_gdp - 2636c3fe-fada-4e46-abb0-14bbbe9dc60f_Data.csv.csv'), skiprows=4)

print("PV shape:", df_pv.shape)
print("GE shape:", df_ge.shape)
print("Inflation/GDP shape:", df_inf_gdp.shape)

print("PV columns:", df_pv.columns.tolist())
print("Inflation/GDP columns:", df_inf_gdp.columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PV shape: (215, 18)
GE shape: (215, 18)
Inflation/GDP shape: (533, 18)

PV columns: ['American Samoa', 'ASM', 'Political Stability and Absence of Violence/Terrorism: Estimate', 'PV.EST', '0.9191467166', '0.9339901805', '0.9529908299', '0.9289857745', '1.08068347', '1.151678085', '1.158555984', '1.184323668', '1.166297317', '1.144933939', '1.09020865', '1.066840887', '1.123173952', '1.114221215']

Inflation/GDP columns: ['Albania', 'ALB', 'GDP growth (annual %)', 'NY.GDP.MKTP.KD.ZG', '2.973154427', '2.463516839', '0.9841298802', '1.707228318', '2.240226954', '2.227704075', '3.908935514', '3.283175933', '3.671419281', '2.062577902', '-3.313756369', '8.969576191', '4.826800623', '4.015417169']


# Ingestion of Macroeconomic & Governance Indicators
To build a holistic risk model, we ingest secondary CSV datasets representing Political Stability (PV), Government Effectiveness (GE), and Macroeconomic volatility (Inflation/GDP). This module focuses on initial file discovery and structural validation of the raw data.

In [ ]:
df_pv = pd.read_csv(os.path.join(folder, 'wb_pv_est - 9f6967d8-0bd5-4748-8a6e-399a111edaba_Data.csv.csv'), skiprows=3)
df_ge = pd.read_csv(os.path.join(folder, 'ge_est - 0a14da3c-c25f-42f2-b620-dea75446bad0_Data.csv.csv'), skiprows=3)
df_inf_gdp = pd.read_csv(os.path.join(folder, 'inflation_gdp - 2636c3fe-fada-4e46-abb0-14bbbe9dc60f_Data.csv.csv'), skiprows=3)

print("PV columns:", df_pv.columns.tolist())
print("PV first 3 rows:")
print(df_pv.head(3))

PV columns: ['Algeria', 'DZA', 'Political Stability and Absence of Violence/Terrorism: Estimate', 'PV.EST', '-1.259367585', '-1.360560656', '-1.325043321', '-1.202371478', '-1.190535188', '-1.090786576', '-1.099741936', '-0.919614315', '-0.8421077132', '-1.055853844', '-0.8478204012', '-0.9924571514', '-0.650652349', '-0.5778852701']

PV first 3 rows:
          Algeria  DZA  \
0  American Samoa  ASM   
1         Andorra  AND   
2          Angola  AGO   

  Political Stability and Absence of Violence/Terrorism: Estimate  PV.EST  \
0  Political Stability and Absence of Violence/Te...               PV.EST   
1  Political Stability and Absence of Violence/Te...               PV.EST   
2  Political Stability and Absence of Violence/Te...               PV.EST   

    -1.259367585   -1.360560656   -1.325043321  -1.202371478   -1.190535188  \
0   0.9191467166   0.9339901805   0.9529908299  0.9289857745     1.08068347   
1    1.278272152    1.302488327    1.290351152    1.28392601    1.28659331

In [ ]:
print(df_pv.head(3))

          Algeria  DZA  \
0  American Samoa  ASM   
1         Andorra  AND   
2          Angola  AGO   

  Political Stability and Absence of Violence/Terrorism: Estimate  PV.EST  \
0  Political Stability and Absence of Violence/Te...               PV.EST   
1  Political Stability and Absence of Violence/Te...               PV.EST   
2  Political Stability and Absence of Violence/Te...               PV.EST   

    -1.259367585   -1.360560656   -1.325043321  -1.202371478   -1.190535188  \
0   0.9191467166   0.9339901805   0.9529908299  0.9289857745     1.08068347   
1    1.278272152    1.302488327    1.290351152    1.28392601    1.286593318   
2  -0.2261823416  -0.3692378104  -0.3893225491  -0.391233474  -0.3332320452   

    -1.090786576   -1.099741936   -0.919614315  -0.8421077132   -1.055853844  \
0    1.151678085    1.158555984    1.184323668    1.166297317    1.144933939   
1    1.365985394    1.382750273    1.392889738    1.391248584     1.57736671   
2  -0.5053858161  -0.32158008

In [ ]:
raw = pd.read_csv(os.path.join(folder, 'wb_pv_est - 9f6967d8-0bd5-4748-8a6e-399a111edaba_Data.csv.csv'), header=None)
raw.head(10)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,Country Name,Country Code,Series Name,Series Code,2010 [YR2010],2011 [YR2011],2012 [YR2012],2013 [YR2013],2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023]
1,Afghanistan,AFG,Political Stability and Absence of Violence/Te...,PV.EST,-2.579151869,-2.502059698,-2.418561459,-2.519349098,-2.411068439,-2.56262517,-2.662156105,-2.794975519,-2.753441334,-2.65253973,-2.702721119,-2.518676043,-2.544955969,-2.484080553
2,Albania,ALB,Political Stability and Absence of Violence/Te...,PV.EST,-0.1914829016,-0.2823794186,-0.1436315924,0.09192978591,0.4859862328,0.3416390419,0.3374478817,0.3737707436,0.3667792678,0.1101050526,0.0886125192,0.1963719279,0.106261313,0.1833489686
3,Algeria,DZA,Political Stability and Absence of Violence/Te...,PV.EST,-1.259367585,-1.360560656,-1.325043321,-1.202371478,-1.190535188,-1.090786576,-1.099741936,-0.919614315,-0.8421077132,-1.055853844,-0.8478204012,-0.9924571514,-0.650652349,-0.5778852701
4,American Samoa,ASM,Political Stability and Absence of Violence/Te...,PV.EST,0.9191467166,0.9339901805,0.9529908299,0.9289857745,1.08068347,1.151678085,1.158555984,1.184323668,1.166297317,1.144933939,1.09020865,1.066840887,1.123173952,1.114221215
5,Andorra,AND,Political Stability and Absence of Violence/Te...,PV.EST,1.278272152,1.302488327,1.290351152,1.28392601,1.286593318,1.365985394,1.382750273,1.392889738,1.391248584,1.57736671,1.58867538,1.58125937,1.585988402,1.583465934
6,Angola,AGO,Political Stability and Absence of Violence/Te...,PV.EST,-0.2261823416,-0.3692378104,-0.3893225491,-0.391233474,-0.3332320452,-0.5053858161,-0.3215800822,-0.3878954947,-0.3472085893,-0.3684116006,-0.5999178886,-0.7104807496,-0.6301085353,-0.3415050507
7,Anguilla,AIA,Political Stability and Absence of Violence/Te...,PV.EST,1.373761654,1.550927639,1.476920247,1.533979535,1.164254904,1.220001817,1.284042716,1.358390927,1.255063415,1.317046165,1.503842711,1.528787374,1.123173952,1.114221215
8,Antigua and Barbuda,ATG,Political Stability and Absence of Violence/Te...,PV.EST,0.9049993753,0.9575882554,0.9757880569,0.9588633776,0.9845507145,0.9875642657,0.8592496514,0.7320609093,0.8231021166,0.9380796552,0.9295788407,0.9496698976,0.938082099,0.9233932495
9,Argentina,ARG,Political Stability and Absence of Violence/Te...,PV.EST,-0.08479786664,0.1589936614,0.1030405462,0.06531426311,-0.005121871363,0.00876413472,0.1977498978,0.1625421643,0.006909812335,-0.09782309085,-0.0719974041,0.0005300586345,-0.09859433025,-0.1275179088


#Data Refactoring: Reshaping for Time-Series Analysis
Raw World Bank data is typically wide (years as columns). This section implements a custom transformation function to melt the data into a long format. This refactoring is essential for performing efficient vectorized operations and relational merges across different indicator sets.

In [ ]:
import numpy as np

def melt_wb(df, value_name):
    year_cols = [c for c in df.columns if 'YR' in c]
    df_long = df.melt(
        id_vars=['Country Name', 'Country Code', 'Series Code'],
        value_vars=year_cols,
        var_name='year',
        value_name=value_name
    )
    df_long['year'] = df_long['year'].str.extract(r'(\d{4})').astype(int)
    df_long[value_name] = pd.to_numeric(df_long[value_name], errors='coerce')
    df_long = df_long.drop(columns='Series Code')
    return df_long

df_pv_long = melt_wb(df_pv, 'political_stability')
df_ge_long = melt_wb(df_ge, 'govt_effectiveness')

df_inflation = df_inf_gdp[df_inf_gdp['Series Code'] == 'FP.CPI.TOTL.ZG'].copy()
df_gdp = df_inf_gdp[df_inf_gdp['Series Code'] == 'NY.GDP.MKTP.KD.ZG'].copy()

df_inflation_long = melt_wb(df_inflation, 'inflation')
df_gdp_long = melt_wb(df_gdp, 'gdp_growth')

df_master = df_pv_long.merge(df_ge_long, on=['Country Name', 'Country Code', 'year'], how='outer')
df_master = df_master.merge(df_inflation_long, on=['Country Name', 'Country Code', 'year'], how='outer')
df_master = df_master.merge(df_gdp_long, on=['Country Name', 'Country Code', 'year'], how='outer')

print("Master shape:", df_master.shape)
print("Columns:", df_master.columns.tolist())
print("Sample:")
print(df_master.head(5))
print("Null counts:")
print(df_master.isnull().sum())

Master shape: (4032, 7)

Columns: ['Country Name', 'Country Code', 'year', 'political_stability', 'govt_effectiveness', 'inflation', 'gdp_growth']

Sample:
  Country Name Country Code  year  political_stability  govt_effectiveness  \
0  Afghanistan          AFG  2010            -2.579152           -1.478316   
1  Afghanistan          AFG  2011            -2.502060           -1.474100   
2  Afghanistan          AFG  2012            -2.418561           -1.375535   
3  Afghanistan          AFG  2013            -2.519349           -1.401463   
4  Afghanistan          AFG  2014            -2.411068           -1.361646   

   inflation  gdp_growth  
0   2.178538   14.362441  
1  11.804186    0.426355  
2   6.441213   12.752287  
3   7.385772    5.600745  
4   4.673996    2.724543  

Null counts:
Country Name            126
Country Code            154
year                      0
political_stability    1063
govt_effectiveness     1074
inflation               771
gdp_growth              428
dty

In [ ]:
df_lpi_2023 = pd.read_csv(os.path.join(folder, 'LPI_worldbank - 2023.csv'))
df_lpi_2023 = df_lpi_2023[['Economy', 'LPI Score']].copy()
df_lpi_2023.columns = ['Country Name', 'lpi_score']
df_lpi_2023['year'] = 2023

old_years = [2018, 2016, 2014, 2012, 2010, 2007]
lpi_old = []

for yr in old_years:
    df = pd.read_csv(
        os.path.join(folder, f'LPI_worldbank - {yr}.csv'),
        header=2
    )
    df = df.iloc[:, [0, 2]].copy()
    df.columns = ['Country Name', 'lpi_score']
    df['year'] = yr
    lpi_old.append(df)

df_lpi = pd.concat([df_lpi_2023] + lpi_old, ignore_index=True)
df_lpi = df_lpi.dropna(subset=['Country Name', 'lpi_score'])
df_lpi['lpi_score'] = pd.to_numeric(df_lpi['lpi_score'], errors='coerce')
df_lpi['Country Name'] = df_lpi['Country Name'].str.strip()

print("LPI shape:", df_lpi.shape)
print(df_lpi['year'].value_counts().sort_index())
print(df_lpi.head(5))

LPI shape: (1079, 3)
year
2007    150
2010    155
2012    155
2014    160
2016    160
2018    160
2023    139
Name: count, dtype: int64
  Country Name  lpi_score  year
0    Singapore        4.3  2023
1      Finland        4.2  2023
2      Denmark        4.1  2023
3      Germany        4.1  2023
4  Netherlands        4.1  2023


In [ ]:
for f in os.listdir(folder):
    print(f)

inflation_gdp - 2636c3fe-fada-4e46-abb0-14bbbe9dc60f_Data.csv.csv
LPI_worldbank - 2018.csv
wb_pv_est - 9f6967d8-0bd5-4748-8a6e-399a111edaba_Data.csv.csv
ge_est - 0a14da3c-c25f-42f2-b620-dea75446bad0_Data.csv.csv
LPI_worldbank - 2023.csv
LPI_worldbank - 2012.csv
LPI_worldbank - 2014.csv
LPI_worldbank - 2010.csv
LPI_worldbank - 2007.csv
LPI_worldbank - 2016.csv


 # Feature Engineering
 1. Multi-Dimensional Risk Modeling:
 This is the core analytical engine of the project. We perform data normalization (scaling indicators to a 0 to 1 range) and compute Inflation Volatility using rolling standard deviations. A weighted Risk Score formula is then applied, synthesizing political, logistical, and economic variables into a single, interpretable metric.
2. Data Integrity & Missing Value Imputation:
Real-world datasets often suffer from temporal gaps. In this stage, we utilize Forward Filling (ffill) to handle missing LPI scores across non-survey years and prune the dataset of null values that would otherwise skew the risk calculations, ensuring the final output is robust.

## Normalizing and forward fill

In [ ]:
# merge/joining lpi into master using left join
df_master = df_master.merge(df_lpi, on=['Country Name', 'year'], how='left')

# Forward filling the gaps
df_master = df_master.sort_values(['Country Name', 'year'])
df_master['lpi_score'] = df_master.groupby('Country Name')['lpi_score'].ffill()

# Normalize function to scales values to range 0 to 1
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

df_master['pv_norm']  = normalize(df_master['political_stability'])
df_master['ge_norm']  = normalize(df_master['govt_effectiveness'])
df_master['lpi_norm'] = normalize(df_master['lpi_score'])

# inflation volatility - rolling std per country
df_master['inf_volatility'] = df_master.groupby('Country Name')['inflation'] \.transform(lambda x: x.rolling(3, min_periods=1).std())
df_master['inf_norm'] = normalize(df_master['inf_volatility'])



Master shape: (4032, 14)

Null counts:
Country Name            126
Country Code            154
year                      0
political_stability    1063
govt_effectiveness     1074
inflation               771
gdp_growth              428
lpi_score              1815
pv_norm                1063
ge_norm                1074
lpi_norm               1815
inf_volatility          996
inf_norm                996
risk_score             2064
dtype: int64

Top 10 riskiest countries (2023):
                  Country Name  political_stability  lpi_score  risk_score
3429                     Sudan            -2.466939       2.40       93.12
13                 Afghanistan            -2.484081       1.90       92.97
2057                     Libya            -2.168015       1.90       87.56
1455                     Haiti            -1.434522       2.10       83.03
1665                      Iraq            -2.413098       2.40       81.68
601   Central African Republic            -2.203107       2.50       80

## Risk score formula

In [ ]:
'''Risk score formula
higher political stability, govt effectiveness, lpi = LOWER risk
higher inflation volatility = HIGHER risk'''

df_master['risk_score'] = (
    (1 - df_master['pv_norm'])  * 0.35 +
    (1 - df_master['ge_norm'])  * 0.25 +
    (1 - df_master['lpi_norm']) * 0.30 +
    df_master['inf_norm']       * 0.10
).round(4)

# scale risk score to 0-100 for readability
df_master['risk_score'] = (normalize(df_master['risk_score']) * 100).round(2)

# check
print("Master shape:", df_master.shape)
print("Null counts:")
print(df_master.isnull().sum())
print("Top 10 riskiest countries (2023):")
result= df_master[df_master['year'] == 2023]
      .sort_values('risk_score', ascending=False)
      [['Country Name', 'political_stability', 'lpi_score', 'risk_score']]
      .head(10)
print(result)

## Dropping rows and reset index

In [ ]:
# drop rows where there is no risk score
df_clean = df_master.dropna(subset=['risk_score']).copy()

# drop rows with no country name or code
df_clean = df_clean.dropna(subset=['Country Name']).copy()

# reset index
df_clean = df_clean.reset_index(drop=True)

# keep only relevant columns
df_final = df_clean[[
    'Country Name',
    'Country Code',
    'year',
    'political_stability',
    'govt_effectiveness',
    'inflation',
    'gdp_growth',
    'lpi_score',
    'risk_score'
]].copy()

print("Final shape:", df_final.shape)
print("Null counts:")
print(df_final.isnull().sum())
print("Years present:")
print(df_final['year'].value_counts().sort_index())
print("Sample:")
print(df_final.head(5))

Final shape: (1968, 9)

Null counts:
Country Name           0
Country Code           0
year                   0
political_stability    0
govt_effectiveness     0
inflation              8
gdp_growth             3
lpi_score              0
risk_score             0
dtype: int64

Years present:
year
2011    142
2012    152
2013    152
2014    153
2015    153
2016    154
2017    154
2018    151
2019    152
2020    152
2021    150
2022    150
2023    153
Name: count, dtype: int64

Sample:
  Country Name Country Code  year  political_stability  govt_effectiveness  \
0  Afghanistan          AFG  2011            -2.502060           -1.474100   
1  Afghanistan          AFG  2012            -2.418561           -1.375535   
2  Afghanistan          AFG  2013            -2.519349           -1.401463   
3  Afghanistan          AFG  2014            -2.411068           -1.361646   
4  Afghanistan          AFG  2015            -2.562625           -1.397395   

   inflation  gdp_growth  lpi_score  risk_sc

# SQL DataBase Integration
To demonstrate production-ready data handling, the processed master dataset is migrated into an SQLite database. This allows for structured querying, better data persistence, and mimics the architecture of professional data engineering environments.

## Verifying the db

In [ ]:
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

print("Row count:")
print(pd.read_sql("SELECT COUNT(*) as total_rows FROM country_risk", conn))

print("top 5 riskiest countries in 2023")
print(pd.read_sql("""
    SELECT "Country Name", "Country Code", risk_score, political_stability, lpi_score
    FROM country_risk
    WHERE year = 2023
    ORDER BY risk_score DESC
    LIMIT 5
""", conn))

Tables in database:
           name
0  country_risk

Row count:
   total_rows
0        1968

Sample query - top 5 riskiest countries in 2023:
  Country Name Country Code  risk_score  political_stability  lpi_score
0        Sudan          SDN       93.12            -2.466939        2.4
1  Afghanistan          AFG       92.97            -2.484081        1.9
2        Libya          LBY       87.56            -2.168015        1.9
3        Haiti          HTI       83.03            -1.434522        2.1
4         Iraq          IRQ       81.68            -2.413098        2.4


## 4 Advance Analytical Querying
Utilizing SQL, we perform deep-dive analysis on the risk data. This includes using Window Functions to rank countries by risk within specific years and Common Table Expressions (CTEs) to identify nations experiencing the most significant risk deterioration over a period of five years from 2018 to 2023.

In [ ]:
# Query 1: Window function - rank countries by risk within each year
print("Query 1 - Risk rankings with window function:")
print(pd.read_sql("""
    SELECT
        "Country Name",
        year,
        risk_score,
        RANK() OVER (PARTITION BY year ORDER BY risk_score DESC) as risk_rank
    FROM country_risk
    WHERE year IN (2018, 2023)
    ORDER BY year, risk_rank
    LIMIT 10
""", conn))

# Query 2: CTE - countries that worsened the most 2018 to 2023
print("Query 2 - Biggest risk deterioration 2018 to 2023:")
print(pd.read_sql("""
    WITH risk_2018 AS (
        SELECT "Country Name", "Country Code", risk_score as score_2018
        FROM country_risk WHERE year = 2018
    ),
    risk_2023 AS (
        SELECT "Country Name", "Country Code", risk_score as score_2023
        FROM country_risk WHERE year = 2023
    )
    SELECT
        r18."Country Name",
        r18.score_2018,
        r23.score_2023,
        ROUND(r23.score_2023 - r18.score_2018, 2) as risk_change
    FROM risk_2018 r18
    JOIN risk_2023 r23 ON r18."Country Code" = r23."Country Code"
    ORDER BY risk_change DESC
    LIMIT 10
""", conn))

# Query 3: Average risk by region proxy - top 10 highest avg risk
print("Query 3 - Most consistently high risk countries (avg across all years):")
print(pd.read_sql("""
    SELECT
        "Country Name",
        ROUND(AVG(risk_score), 2) as avg_risk,
        ROUND(MIN(risk_score), 2) as best_year_risk,
        ROUND(MAX(risk_score), 2) as worst_year_risk,
        COUNT(*) as years_present
    FROM country_risk
    GROUP BY "Country Name"
    HAVING years_present >= 5
    ORDER BY avg_risk DESC
    LIMIT 10
""", conn))

# Query 4: Alert trigger - countries crossing risk threshold
print("Query 4 - High risk alert log (risk score above 70):")
print(pd.read_sql("""
    SELECT
        "Country Name",
        year,
        risk_score,
        political_stability,
        lpi_score,
        CASE
            WHEN risk_score >= 85 THEN 'Critical'
            WHEN risk_score >= 70 THEN 'High'
            ELSE 'Moderate'
        END as risk_level
    FROM country_risk
    WHERE risk_score >= 70
    AND year = 2023
    ORDER BY risk_score DESC
""", conn))

Query 1 - Risk rankings with window function:
               Country Name  year  risk_score  risk_rank
0               Afghanistan  2018       91.15          1
1      Syrian Arab Republic  2018       88.85          2
2                     Libya  2018       87.81          3
3  Central African Republic  2018       85.10          4
4                      Iraq  2018       84.57          5
5                   Burundi  2018       79.52          6
6                     Sudan  2018       78.81          7
7                   Nigeria  2018       75.58          8
8                  Pakistan  2018       75.28          9
9                      Chad  2018       74.60         10

Query 2 - Biggest risk deterioration 2018 to 2023:
         Country Name  score_2018  score_2023  risk_change
0               Sudan       78.81       93.12        14.31
1        Burkina Faso       62.28       76.26        13.98
2  Iran, Islamic Rep.       60.94       74.31        13.37
3             Belarus       48.59      

## Automated Alert Logic & Final Export
The final stage categorizes countries into categorical risk levels (Critical, High, Moderate, Low) based on the calculated scores. The resulting "Alert Log" is then exported as a finalized CSV, optimized for downstream consumption in visualization tools like Power BI or Tableau.

In [ ]:
alert_log = pd.read_sql("""
    SELECT
        "Country Name",
        "Country Code",
        year,
        risk_score,
        political_stability,
        govt_effectiveness,
        lpi_score,
        inflation,
        gdp_growth,
        CASE
            WHEN risk_score >= 85 THEN 'Critical'
            WHEN risk_score >= 70 THEN 'High'
            WHEN risk_score >= 50 THEN 'Moderate'
            ELSE 'Low'
        END as risk_level
    FROM country_risk
    ORDER BY risk_score DESC
""", conn)

alert_log.to_sql('alert_log', conn, if_exists='replace', index=False)

export_path = '/content/drive/MyDrive/4files_world_bank/supply_chain_risk_final.csv'
alert_log.to_csv(export_path, index=False)

print("Alert log shape:", alert_log.shape)
print("Risk level distribution:")
print(alert_log['risk_level'].value_counts())
print("Tables in database:")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))
print("Exported to:", export_path)

conn.close()

Alert log shape: (1968, 10)

Risk level distribution:
risk_level
Low         959
Moderate    809
High        156
Critical     44
Name: count, dtype: int64

Tables in database:
           name
0  country_risk
1     alert_log

Exported to: /content/drive/MyDrive/4files_world_bank/supply_chain_risk_final.csv
